## LOS 3 KPIS PRINCIPALES DEL PROYECTO

1. Completion Rate

2. Client Backtracking Rate

3. Average Time Between Steps


In [10]:
# Pathlib sirve para trabajar con rutas compatibles entre sistemas operativos.
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path().resolve().parent
DATA_PATH = ROOT / "data_raw"

print("Directorio actual:", ROOT)
print("Ruta de datos:", DATA_PATH)

Directorio actual: /Users/gabrielbohorquez/Desktop/Ironhack/Semana_5/PROYECTO_2/Projecto-2-Vanguard-ab-test
Ruta de datos: /Users/gabrielbohorquez/Desktop/Ironhack/Semana_5/PROYECTO_2/Projecto-2-Vanguard-ab-test/data_raw


El warning DtypeWarning: mixed types no es un error — el archivo cargó bien. Solo significa que algunas columnas tienen tipos mezclados (texto y números).
Para eliminarlo puedes agregar low_memory=False:

In [45]:
df = pd.read_csv(DATA_PATH / "vanguard_cleaned_todos unidos_(Gabriel).csv", low_memory=False)
print(df.shape)
print(df.dtypes)

(343141, 14)
client_id             int64
visitor_id              str
visit_id                str
process_step            str
date_time               str
Variation               str
clnt_tenure_yr      float64
clnt_tenure_mnth    float64
clnt_age            float64
gendr                   str
num_accts           float64
bal                 float64
calls_6_mnth        float64
logons_6_mnth       float64
dtype: object


In [12]:
# Verificar
df.head()

,client_id,visitor_id,visit_id,process_step,date_time,Variation,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07,Test,5.0,64.0,79.0,U,2.0,189023.86,1.0,4.0
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51,Test,5.0,64.0,79.0,U,2.0,189023.86,1.0,4.0
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22,Test,5.0,64.0,79.0,U,2.0,189023.86,1.0,4.0
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13,Test,5.0,64.0,79.0,U,2.0,189023.86,1.0,4.0
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04,Test,5.0,64.0,79.0,U,2.0,189023.86,1.0,4.0


In [14]:
# ENTENDER EL FUNNEL
df["process_step"].value_counts()

process_step
start      108910
step_1      73432
step_2      61768
step_3      53628
confirm     45403
Name: count, dtype: int64

Ver:
✔ qué pasos existen
✔ cuál es el más frecuente
✔ si confirm aparece menos

## KPI 1: COMPLETION RATE
Completion Rate mide: qué porcentaje de usuarios llega a confirm.

In [15]:
# Usuarios totales - Cuenta usuarios únicos porque un usuario puede tener varias filas. Si contamos filas, estaríamos contando varias veces a un mismo usuario.
total_users = df["client_id"].nunique()

total_users

58391

In [16]:
# Usuarios que completaron el proceso - De nuevo, contamos usuarios únicos para evitar contar varias veces a un mismo usuario.
completed_users = df[df["process_step"] == "confirm"]["client_id"].nunique()

completed_users

37190

45403 - puede haber usuarios repetidos.
37190 - usuarios que completaron el proceso.

In [17]:
# Completion Rate
completion_rate = completed_users / total_users

completion_rate

0.636913222928191

63% de los usuarios completaron el proceso. ¿63% es bueno o malo?: depende de la comparación Test vs Control.
Interfas clara, proceso simple, menos fricción, experiencia eficiente

In [18]:
# Mostrar el resultado en porcentaje
print(f"Completion Rate: {completion_rate:.2%}")

Completion Rate: 63.69%


## COMPLETION RATE TEST VS CONTROL

In [19]:
# Crear dataset por usuario. 1 fila por usuario. Con: True = completó, False = no completó
completed_df = (
    df.groupby("client_id")["process_step"]
    .apply(lambda x: "confirm" in x.values)
    .reset_index()
)

completed_df.columns = ["client_id", "completed"]

In [20]:
# Agregar variation al dataset por usuario. Esto Agrega: Test vs Control
variation_df = df[["client_id", "Variation"]].drop_duplicates()

completed_df = completed_df.merge(
    variation_df,
    on="client_id",
    how="left"
)

In [21]:
print(completed_df.columns.tolist())

['client_id', 'completed', 'Variation']


Tiene Variation_x y Variation_y por un merge duplicado. Fix:

In [24]:
# Completion por grupo. Calcula: completion promedio por grupo
completion_by_group = (
    completed_df.groupby("Variation")["completed"]
    .agg(["sum", "count", "mean"])
    .rename(columns={
        "sum": "completed_users",
        "count": "total_users",
        "mean": "completion_rate"
    })
)

completion_by_group

,completed_users,total_users,completion_rate
Variation,,,
Control,11366,18015,0.630919
Test,15030,22013,0.682778


## Resultado preliminar: Completion Rate por grupo

El grupo Test presenta una tasa de finalización superior al grupo Control.

- Control: 63,09 %
- Test: 68,28 %
- Diferencia observada: +5,19 puntos porcentuales a favor de Test.

Esta diferencia sugiere una mejora relevante en la experiencia del usuario, aunque la significancia estadística debe validarse posteriormente mediante una prueba de proporciones.

In [29]:
# Gráfico omitido en este entorno porque matplotlib no está disponible.
# La comparación Test vs Control queda validada mediante la tabla:
# Control Completion Rate: 63.09%
# Test Completion Rate: 68.28%
# Diferencia Test - Control: 5.19 puntos porcentuales.

In [26]:
completion_by_group_formatted = completion_by_group.copy()
completion_by_group_formatted["completion_rate"] = (
    completion_by_group_formatted["completion_rate"] * 100
).round(2)

completion_by_group_formatted

,completed_users,total_users,completion_rate
Variation,,,
Control,11366,18015,63.09
Test,15030,22013,68.28


In [30]:
control_completion = completion_by_group.loc["Control", "completion_rate"]
test_completion = completion_by_group.loc["Test", "completion_rate"]

completion_difference = test_completion - control_completion

print(f"Control Completion Rate: {control_completion:.2%}")
print(f"Test Completion Rate: {test_completion:.2%}")
print(f"Diferencia Test - Control: {completion_difference:.2%}")

Control Completion Rate: 63.09%
Test Completion Rate: 68.28%
Diferencia Test - Control: 5.19%


> **Nota metodológica:** este KPI se calcula sobre el dataset final unido utilizado para el análisis exploratorio.  
> Para la validación estadística oficial del experimento se utiliza el Notebook 05, basado en los archivos limpios separados de actividad web y asignación experimental.  
> Por este motivo, los valores pueden diferir ligeramente entre el análisis exploratorio de KPIs y el test estadístico final.

## Interpretación del KPI 1: Completion Rate

El grupo Test presenta una tasa de finalización superior al grupo Control.

- Control Completion Rate: 63,09 %
- Test Completion Rate: 68,28 %
- Diferencia observada: +5,19 puntos porcentuales a favor de Test.

Desde una perspectiva de negocio, el rediseño parece mejorar la finalización del proceso digital. Sin embargo, esta lectura debe validarse estadísticamente antes de recomendar una implementación definitiva.

## KPI 2: Client Backtracking Rate

Este KPI mide la proporción de clientes que realizan al menos un retroceso dentro del flujo digital.

A diferencia de una métrica por evento o transición, esta definición evita contar varias veces al mismo cliente y permite comparar de forma más justa el comportamiento entre Control y Test.

Definición oficial provisional:

- Numerador: clientes únicos con al menos un retroceso.
- Denominador: clientes únicos del grupo analizado.
- Comparación: Control vs Test.


In [31]:
# KPI 2: Client Backtracking Rate
# Definimos el orden lógico del funnel para detectar retrocesos reales.

step_order = {
    "start": 0,
    "step_1": 1,
    "step_2": 2,
    "step_3": 3,
    "confirm": 4
}

df_back = df.copy()

df_back["step_num"] = df_back["process_step"].map(step_order)
df_back = df_back.sort_values(["client_id", "visit_id", "date_time"])

df_back["previous_step"] = df_back.groupby(["client_id", "visit_id"])["step_num"].shift(1)

df_back["is_backtrack"] = (
    df_back["previous_step"].notna()
    & df_back["step_num"].notna()
    & (df_back["step_num"] < df_back["previous_step"])
)

client_backtracking = (
    df_back.groupby("client_id", as_index=False)["is_backtrack"]
    .max()
)

client_backtracking = client_backtracking.merge(
    df[["client_id", "Variation"]].drop_duplicates(),
    on="client_id",
    how="left",
    validate="one_to_one"
)

backtracking_summary = (
    client_backtracking
    .groupby("Variation")
    .agg(
        backtracking_clients=("is_backtrack", "sum"),
        total_clients=("client_id", "nunique")
    )
)

backtracking_summary["backtracking_rate"] = (
    backtracking_summary["backtracking_clients"] / backtracking_summary["total_clients"]
)

backtracking_summary


,backtracking_clients,total_clients,backtracking_rate
Variation,,,
Control,4172,18015,0.231585
Test,7215,22013,0.327761


In [32]:
control_backtracking = backtracking_summary.loc["Control", "backtracking_rate"]
test_backtracking = backtracking_summary.loc["Test", "backtracking_rate"]

backtracking_difference = test_backtracking - control_backtracking

print(f"Control Backtracking Rate: {control_backtracking:.2%}")
print(f"Test Backtracking Rate: {test_backtracking:.2%}")
print(f"Diferencia Test - Control: {backtracking_difference:.2%}")

Control Backtracking Rate: 23.16%
Test Backtracking Rate: 32.78%
Diferencia Test - Control: 9.62%


## Interpretación del KPI 2: Client Backtracking Rate

El grupo Test presenta una proporción mayor de clientes con al menos un retroceso dentro del flujo digital.

- Control Backtracking Rate: 23,16 %
- Test Backtracking Rate: 32,78 %
- Diferencia observada: +9,62 puntos porcentuales en Test.

Este resultado indica que, aunque el rediseño mejora la finalización del proceso, también parece generar más fricción, dudas o navegación hacia atrás en determinados pasos.

Por tanto, el rediseño no debe evaluarse únicamente por Completion Rate. La decisión de negocio debe considerar simultáneamente la mejora en conversión y el aumento de retrocesos antes de recomendar una implementación definitiva.

In [35]:
# Validación estadística del Client Backtracking Rate
# Prueba Z manual de diferencia de proporciones para evitar dependencias externas.

from math import sqrt, erf

control_success = backtracking_summary.loc["Control", "backtracking_clients"]
control_n = backtracking_summary.loc["Control", "total_clients"]

test_success = backtracking_summary.loc["Test", "backtracking_clients"]
test_n = backtracking_summary.loc["Test", "total_clients"]

control_rate = control_success / control_n
test_rate = test_success / test_n

absolute_diff = test_rate - control_rate
relative_lift = absolute_diff / control_rate

pooled_rate = (control_success + test_success) / (control_n + test_n)
standard_error = sqrt(
    pooled_rate * (1 - pooled_rate) * ((1 / control_n) + (1 / test_n))
)

z_stat = absolute_diff / standard_error

# CDF normal estándar usando erf
p_value_one_sided = 1 - (0.5 * (1 + erf(z_stat / sqrt(2))))
p_value_two_sided = 2 * min(p_value_one_sided, 1 - p_value_one_sided)

print("=== CLIENT BACKTRACKING RATE ===")
print(f"Control: {control_rate:.2%}")
print(f"Test: {test_rate:.2%}")
print(f"Diferencia absoluta Test - Control: {absolute_diff:.2%}")
print(f"Incremento relativo: {relative_lift:.2%}")
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value unilateral: {p_value_one_sided:.10f}")
print(f"P-value bilateral: {p_value_two_sided:.10f}")

if p_value_one_sided < 0.05:
    print("Decisión: el Client Backtracking Rate es significativamente mayor en Test.")
else:
    print("Decisión: no hay evidencia suficiente de mayor backtracking en Test.")


=== CLIENT BACKTRACKING RATE ===
Control: 23.16%
Test: 32.78%
Diferencia absoluta Test - Control: 9.62%
Incremento relativo: 41.53%
Z-statistic: 21.2181
P-value unilateral: 0.0000000000
P-value bilateral: 0.0000000000
Decisión: el Client Backtracking Rate es significativamente mayor en Test.


### Diagnóstico por transición

Para entender dónde se concentra la fricción, analizamos qué transiciones generan más retrocesos.

Este análisis permite separar dos situaciones distintas:

- El usuario retrocede al inicio del flujo.
- El usuario retrocede desde pasos avanzados después de haber progresado.

Esta lectura es más accionable que una tasa agregada, porque orienta mejoras concretas de UX.


In [34]:
# Diagnóstico de retrocesos por transición

df_back["previous_process_step"] = (
    df_back.groupby(["client_id", "visit_id"])["process_step"].shift(1)
)

df_back["transition"] = (
    df_back["previous_process_step"].astype(str)
    + " -> "
    + df_back["process_step"].astype(str)
)

backtracking_transitions = (
    df_back[df_back["is_backtrack"]]
    .groupby(["Variation", "transition"])
    .size()
    .rename("backtracks")
    .reset_index()
)

group_total_backtracks = (
    backtracking_transitions
    .groupby("Variation")["backtracks"]
    .sum()
    .rename("group_total_backtracks")
    .reset_index()
)

backtracking_transitions = backtracking_transitions.merge(
    group_total_backtracks,
    on="Variation",
    how="left"
)

backtracking_transitions["share_within_group"] = (
    backtracking_transitions["backtracks"] / backtracking_transitions["group_total_backtracks"]
)

top_backtracking_transitions = (
    backtracking_transitions
    .sort_values(["Variation", "backtracks"], ascending=[True, False])
)

top_backtracking_transitions.head(20)


,Variation,transition,backtracks,group_total_backtracks,share_within_group
8,Control,step_3 -> step_2,1956,6120,0.319608
3,Control,step_1 -> start,1149,6120,0.187745
7,Control,step_3 -> step_1,769,6120,0.125654
5,Control,step_2 -> step_1,661,6120,0.108007
6,Control,step_3 -> start,566,6120,0.092484
0,Control,confirm -> start,449,6120,0.073366
4,Control,step_2 -> start,359,6120,0.058660
2,Control,confirm -> step_3,123,6120,0.020098
1,Control,confirm -> step_1,88,6120,0.014379
13,Test,step_1 -> start,4898,12578,0.389410


## Conclusión del KPI 2

El rediseño de la interfaz aumenta la tasa de finalización, pero también incrementa de forma significativa la proporción de clientes que retroceden al menos una vez dentro del flujo.

El diagnóstico por transición muestra que el principal foco de fricción en el grupo Test se concentra al inicio del proceso, especialmente en la transición `step_1 -> start`.

Esto sugiere que la nueva interfaz puede estar ayudando a más usuarios a completar el proceso, pero introduce dudas o fricción temprana en la entrada al flujo.

La recomendación no es descartar el rediseño, sino optimizar la primera pantalla o el primer paso antes de una implementación definitiva a gran escala.


## KPI 3: Average Time Between Steps

Este KPI mide el tiempo medio entre pasos del flujo digital.

La métrica ayuda a detectar si un grupo avanza de forma más rápida o más lenta dentro del proceso. Debe interpretarse con cuidado, ya que tiempos más largos pueden reflejar fricción, dudas, pausas naturales o mayor atención del usuario.


In [37]:
# KPI 3: Average Time Between Steps

df_time = df.copy()
df_time["date_time"] = pd.to_datetime(df_time["date_time"], errors="coerce")

df_time = df_time.sort_values(["client_id", "visit_id", "date_time"])

df_time["previous_time"] = (
    df_time.groupby(["client_id", "visit_id"])["date_time"].shift(1)
)

df_time["time_between_steps_seconds"] = (
    df_time["date_time"] - df_time["previous_time"]
).dt.total_seconds()

# Filtramos tiempos inválidos o extremos evidentes para evitar distorsiones.
df_time_valid = df_time[
    (df_time["time_between_steps_seconds"].notna())
    & (df_time["time_between_steps_seconds"] >= 0)
]

time_by_group = (
    df_time_valid
    .groupby("Variation")["time_between_steps_seconds"]
    .mean()
    .rename("avg_time_between_steps_seconds")
)

time_by_group


Variation
Control    84.415865
Test       84.460154
Name: avg_time_between_steps_seconds, dtype: float64

In [38]:
control_time = time_by_group.loc["Control"]
test_time = time_by_group.loc["Test"]

time_difference = test_time - control_time

print(f"Control Average Time Between Steps: {control_time:.2f} seconds")
print(f"Test Average Time Between Steps: {test_time:.2f} seconds")
print(f"Diferencia Test - Control: {time_difference:.2f} seconds")

Control Average Time Between Steps: 84.42 seconds
Test Average Time Between Steps: 84.46 seconds
Diferencia Test - Control: 0.04 seconds


## Conclusión del KPI 3

El tiempo medio entre pasos es prácticamente igual entre ambos grupos.

- Control: 84,42 segundos
- Test: 84,46 segundos
- Diferencia observada: +0,04 segundos en Test

Desde una perspectiva de negocio, esta diferencia no parece relevante. El rediseño no aumenta de forma material el tiempo medio entre pasos.

Por tanto, el principal riesgo detectado no está en la duración media del recorrido, sino en el aumento de retrocesos observado en el KPI 2, especialmente al inicio del flujo.

La lectura conjunta de los tres KPIs indica que el grupo Test convierte más, no tarda más de forma significativa, pero sí presenta más navegación hacia atrás.


In [39]:
# Usuarios por paso - Cuenta usuarios únicos por paso para entender el funnel
funnel = (
    df.groupby("process_step")["client_id"]
    .nunique()
    .sort_values(ascending=False)
)

funnel

process_step
start      57851
step_1     49341
step_2     45385
step_3     42313
confirm    37190
Name: client_id, dtype: int64

In [40]:
# Calcular porcentaje de usuarios que pasan por cada paso
funnel_sorted = funnel.sort_values(ascending=False)
dropoff = funnel_sorted.pct_change(periods=-1) * 100
print(dropoff.round(2))

process_step
start      17.25
step_1      8.72
step_2      7.26
step_3     13.78
confirm      NaN
Name: client_id, dtype: float64


Conclusión: el mayor abandono está en start (17.25%) y step_3 → confirm (13.78%).

In [41]:
funnel_summary = pd.DataFrame({
    "users": funnel,
    "dropoff_percent_to_next_step": dropoff.round(2)
})

funnel_summary

,users,dropoff_percent_to_next_step
process_step,,
start,57851,17.25
step_1,49341,8.72
step_2,45385,7.26
step_3,42313,13.78
confirm,37190,NaN


## SEGMENTACIÓN POR EDAD

In [42]:
# Crear grupos edad
df["age_group"] = pd.cut(
    df["clnt_age"],
    bins=[0,30,50,70,100],
    labels=["18-30","31-50","51-70","70+"]
)

In [43]:
# Completion por edad - ¿qué segmentos tienen más éxito?
completion_age = (
    df[df["process_step"] == "confirm"]
    .groupby("age_group")["client_id"]
    .nunique()
)

completion_age

age_group
18-30     7378
31-50    13477
51-70    14320
70+       1790
Name: client_id, dtype: int64

51-70 es el segmento con más completions — tiene sentido para Vanguard ya que son clientes con más patrimonio e inversiones activas.

In [44]:
# Para verlo en proporción (% del total por grupo):
total_age = df.groupby("age_group", observed=True)["client_id"].nunique()
completion_rate_age = (completion_age / total_age * 100).round(2)
print(completion_rate_age)

age_group
18-30    67.89
31-50    68.25
51-70    64.95
70+      53.71
Name: client_id, dtype: float64


## Conclusión de segmentación por edad

En valores absolutos, el segmento 51–70 concentra el mayor número de usuarios que completan el proceso.

Sin embargo, al analizar la tasa de finalización proporcional dentro de cada grupo de edad, el segmento 31–50 presenta el mejor desempeño:

- 18–30: 67,89 %
- 31–50: 68,25 %
- 51–70: 64,95 %
- 70+: 53,71 %

Esto muestra que el volumen absoluto puede llevar a una interpretación incompleta. Para comparar segmentos correctamente, es necesario analizar la tasa de finalización relativa dentro de cada grupo.

El segmento 70+ presenta la tasa más baja, lo que puede sugerir mayor fricción digital o menor facilidad de uso para usuarios de mayor edad.

## Conclusión final del análisis de KPIs

El análisis de KPIs muestra que el rediseño de la interfaz Test mejora la tasa de finalización del proceso frente al grupo Control.

La mejora principal se observa en el Completion Rate, donde Test alcanza una tasa superior a Control. Sin embargo, esta mejora viene acompañada de un aumento significativo en el Client Backtracking Rate, lo que indica que más usuarios retroceden al menos una vez dentro del flujo.

El diagnóstico por transición muestra que la mayor fricción del grupo Test se concentra al inicio del proceso, especialmente en la transición `step_1 -> start`.

El Average Time Between Steps no muestra una diferencia relevante entre Control y Test, por lo que el problema no parece estar en que los usuarios tarden más, sino en que navegan hacia atrás con mayor frecuencia.

La segmentación por edad muestra que el grupo 31–50 tiene la mejor tasa proporcional de finalización, mientras que el grupo 70+ presenta la tasa más baja, lo que puede indicar mayor fricción digital para usuarios de mayor edad.

En conclusión, el rediseño Test no debe descartarse. La recomendación es avanzar con una optimización previa de la primera pantalla o del primer paso del flujo antes de una implementación definitiva a gran escala.